# Classification Algorithms

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = sns.load_dataset("titanic")

print("Dataset: Titanic")
print(f"Shape: {df.shape}")
print(f"\nTarget: survived (0 = Did not survive, 1 = Survived)")
print(f"\nFeatures:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col} — dtype: {df[col].dtype}")

## EDA AND DATA CLEANING

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
# Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
# Target Distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='survived', data=df, palette=['#e74c3c', '#2ecc71'])
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.title('Survival Distribution')
plt.xlabel('Survived')
plt.ylabel('Count')
plt.xticks([0, 1], ['Did Not Survive (0)', 'Survived (1)'])
plt.tight_layout()
plt.show()

print(f"Survived: {df['survived'].sum()} ({df['survived'].mean()*100:.1f}%)")
print(f"Did Not Survive: {(df['survived']==0).sum()} ({(1-df['survived'].mean())*100:.1f}%)")

In [ ]:
# Survival by Gender, Pclass, Embarked
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.countplot(x='sex', hue='survived', data=df, palette=['#e74c3c', '#2ecc71'], ax=axes[0])
axes[0].set_title('Survival by Gender')

sns.countplot(x='pclass', hue='survived', data=df, palette=['#e74c3c', '#2ecc71'], ax=axes[1])
axes[1].set_title('Survival by Passenger Class')

sns.countplot(x='embarked', hue='survived', data=df, palette=['#e74c3c', '#2ecc71'], ax=axes[2])
axes[2].set_title('Survival by Embarkation Port')

plt.tight_layout()
plt.show()

In [ ]:
# Age Distribution by Survival
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='age', hue='survived', kde=True, bins=40, palette=['#e74c3c', '#2ecc71'])
plt.title('Age Distribution by Survival')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Drop redundant columns
df.drop(['deck', 'embark_town', 'alive', 'who', 'class', 'adult_male'], axis=1, inplace=True)
print(f"Columns after dropping: {list(df.columns)}")

In [ ]:
# Fill missing values
df['age'].fillna(df['age'].mean(), inplace=True)
df.dropna(subset=['embarked'], inplace=True)

print(f"Missing values after cleaning:")
print(df.isnull().sum())
print(f"\nFinal shape: {df.shape}")

In [ ]:
# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])        # female=0, male=1
df['embarked'] = le.fit_transform(df['embarked'])  # C=0, Q=1, S=2
df['alone'] = df['alone'].astype(int)

df.head()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## FEATURE SCALING & TRAIN-TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"Target distribution in train: {dict(y_train.value_counts())}")
print(f"Target distribution in test: {dict(y_test.value_counts())}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# Helper function to evaluate models
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    print(f"--- {name} ---")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print()
    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}

In [ ]:
# Store all results for comparison
results = []

## 1. LOGISTIC REGRESSION

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)
res = evaluate_model('Logistic Regression', lr, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 2. K-NEAREST NEIGHBORS (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
res = evaluate_model('KNN Classifier', knn, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 3. NAIVE BAYES

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
res = evaluate_model('Naive Bayes', nb, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 4. DECISION TREE CLASSIFIER

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
res = evaluate_model('Decision Tree', dt, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 5. SUPPORT VECTOR MACHINE (SVM)

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
res = evaluate_model('SVM (RBF Kernel)', svm, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 6. RANDOM FOREST CLASSIFIER

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
res = evaluate_model('Random Forest', rf, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 7. GRADIENT BOOSTING CLASSIFIER

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
res = evaluate_model('Gradient Boosting', gb, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 8. ADABOOST CLASSIFIER

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
res = evaluate_model('AdaBoost', ada, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 9. XGBOOST CLASSIFIER

In [ ]:
!pip install xgboost -q

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0, eval_metric='logloss')
res = evaluate_model('XGBoost', xgb, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## 10. EXTRA TREES CLASSIFIER

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(n_estimators=100, max_depth=10, random_state=42)
res = evaluate_model('Extra Trees', et, X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

## MODEL COMPARISON

In [ ]:
# Create comparison DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
results_df.index = results_df.index + 1  # Start index from 1 (rank)
results_df.index.name = 'Rank'
results_df

In [ ]:
# Accuracy Comparison Bar Chart
plt.figure(figsize=(14, 7))
colors = sns.color_palette('viridis', len(results_df))
bars = plt.barh(results_df['Model'], results_df['Accuracy'], color=colors)
plt.xlabel('Accuracy')
plt.title('Accuracy Comparison of Classification Models')
plt.xlim(0.5, 1.0)

# Add value labels on bars
for bar, val in zip(bars, results_df['Accuracy']):
    plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# F1 Score Comparison Bar Chart
results_df_f1 = results_df.sort_values('F1', ascending=False)

plt.figure(figsize=(14, 7))
colors = sns.color_palette('magma', len(results_df_f1))
bars = plt.barh(results_df_f1['Model'], results_df_f1['F1'], color=colors)
plt.xlabel('F1 Score')
plt.title('F1 Score Comparison of Classification Models')
plt.xlim(0.5, 1.0)

# Add value labels on bars
for bar, val in zip(bars, results_df_f1['F1']):
    plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## CONFUSION MATRIX — BEST MODEL

In [ ]:
# Identify the best model
best_model_name = results_df.iloc[0]['Model']
print(f"Best Model: {best_model_name}")
print(f"Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")
print(f"F1 Score: {results_df.iloc[0]['F1']:.4f}")

In [ ]:
# Confusion Matrix Heatmap for the best model
best_models = {
    'Logistic Regression': lr,
    'KNN Classifier': knn,
    'Naive Bayes': nb,
    'Decision Tree': dt,
    'SVM (RBF Kernel)': svm,
    'Random Forest': rf,
    'Gradient Boosting': gb,
    'AdaBoost': ada,
    'XGBoost': xgb,
    'Extra Trees': et
}

best = best_models[best_model_name]
y_pred_best = best.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Did Not Survive', 'Survived'],
            yticklabels=['Did Not Survive', 'Survived'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix — {best_model_name}')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report for the Best Model
print(f"Classification Report — {best_model_name}")
print("=" * 55)
print(classification_report(y_test, y_pred_best, target_names=['Did Not Survive', 'Survived']))

## ROC CURVE — TOP 5 MODELS

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(10, 7))

# Top 5 models by accuracy
top5_names = results_df.head(5)['Model'].tolist()
colors_roc = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for name, color in zip(top5_names, colors_roc):
    model = best_models[name]
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.decision_function(X_test_scaled)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Top 5 Models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## HYPERPARAMETER TUNING (Best Model)

In [ ]:
from sklearn.model_selection import GridSearchCV

# GridSearchCV on Gradient Boosting (typically one of the top performers)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1]
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

gb_grid.fit(X_train_scaled, y_train)

print(f"\nBest Parameters: {gb_grid.best_params_}")
print(f"Best CV Accuracy: {gb_grid.best_score_:.4f}")

In [ ]:
# Evaluate the tuned model
y_pred_tuned = gb_grid.predict(X_test_scaled)

acc_tuned = accuracy_score(y_test, y_pred_tuned)
prec_tuned = precision_score(y_test, y_pred_tuned)
rec_tuned = recall_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned)

print(f"--- Tuned Gradient Boosting ---")
print(f"  Accuracy  : {acc_tuned:.4f}")
print(f"  Precision : {prec_tuned:.4f}")
print(f"  Recall    : {rec_tuned:.4f}")
print(f"  F1 Score  : {f1_tuned:.4f}")
print()
print(classification_report(y_test, y_pred_tuned, target_names=['Did Not Survive', 'Survived']))

## CROSS-VALIDATION SCORES

In [ ]:
from sklearn.model_selection import cross_val_score

cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, random_state=42, verbosity=0, eval_metric='logloss')
}

print(f"{'Model':<25} {'Mean Accuracy':>15} {'Std':>10}")
print('-' * 52)

for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"{name:<25} {scores.mean():>15.4f} {scores.std():>10.4f}")

## FEATURE IMPORTANCE (Random Forest)

In [ ]:
# Feature Importance from Random Forest
importances = rf.feature_importances_
feature_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feature_imp.plot(kind='barh', color='teal')
plt.title('Feature Importance — Random Forest Classifier')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## SUMMARY

### Classification Algorithms Covered:
1. **Logistic Regression** — Linear baseline for binary classification
2. **K-Nearest Neighbors (KNN)** — Distance-based instance learning
3. **Naive Bayes** — Probabilistic classifier (Gaussian)
4. **Decision Tree** — Tree-based non-linear model
5. **Support Vector Machine (SVM)** — Kernel-based margin maximizer
6. **Random Forest** — Ensemble of decision trees (bagging)
7. **Gradient Boosting** — Sequential boosting ensemble
8. **AdaBoost** — Adaptive boosting
9. **XGBoost** — Extreme Gradient Boosting
10. **Extra Trees** — Extremely Randomized Trees

### Key Takeaways:
- Ensemble methods (Random Forest, Gradient Boosting, XGBoost) generally outperform simple models
- Feature scaling is crucial for distance-based models (KNN, SVM) and convergence-sensitive models (Logistic Regression)
- Gender (sex) and Passenger Class (pclass) are the most important features for predicting survival
- Hyperparameter tuning via GridSearchCV can further improve model performance
- ROC-AUC provides a threshold-independent view of model discrimination ability